# G. Weight Initialization & H. Activation Functions

**G**: G1-G3 | **H**: H1-H4

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from network import Network, NetworkConfig
from utils import relu, relu_prime, sigmoid, sigmoid_prime, softmax

## G1: No Zero Initialization

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)

for i, w in enumerate(net.weights):
    is_zero = np.allclose(w, 0)
    print(f'Layer {i} all-zero: {is_zero} | {"FAIL" if is_zero else "PASS"}')

## G2: He Uniform / Xavier Uniform Distribution

In [ ]:
np.random.seed(42)
net_he = Network(NetworkConfig(layers=[100, 50, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform'))
net_xav = Network(NetworkConfig(layers=[100, 50, 2], activation='sigmoid',
    loss='cross_entropy', output_activation='softmax', weights_initializer='xavierUniform'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

w_he = net_he.weights[0]; he_lim = np.sqrt(6/100)
axes[0].hist(w_he.flatten(), bins=50)
axes[0].axvline(he_lim, color='r', ls='--', label=f'+/-{he_lim:.3f}')
axes[0].axvline(-he_lim, color='r', ls='--')
axes[0].set_title('He Uniform (fan_in=100)'); axes[0].legend()
ok_he = np.all(np.abs(w_he) <= he_lim + 1e-10)
print(f'He Uniform within range: {ok_he} {"PASS" if ok_he else "FAIL"}')

w_xav = net_xav.weights[0]; xav_lim = np.sqrt(6/(100+50))
axes[1].hist(w_xav.flatten(), bins=50)
axes[1].axvline(xav_lim, color='r', ls='--', label=f'+/-{xav_lim:.3f}')
axes[1].axvline(-xav_lim, color='r', ls='--')
axes[1].set_title('Xavier Uniform (100+50)'); axes[1].legend()
ok_xav = np.all(np.abs(w_xav) <= xav_lim + 1e-10)
print(f'Xavier Uniform within range: {ok_xav} {"PASS" if ok_xav else "FAIL"}')

plt.tight_layout(); plt.show()

## G3: Bias Initialized to Zero

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)

for i, b in enumerate(net.biases):
    ok = np.allclose(b, 0)
    print(f'Layer {i} bias zero: {ok} {"PASS" if ok else "FAIL"}')

## H1: ReLU Forward & Derivative

In [ ]:
x = np.array([-2, -1, 0, 1, 2], dtype=float)

y = relu(x)
exp_y = np.array([0, 0, 0, 1, 2], dtype=float)
print(f'ReLU({x}) = {y}')
print(f'Expected:    {exp_y}')
print(f'Forward: {"PASS" if np.array_equal(y, exp_y) else "FAIL"}')

dy = relu_prime(x)
exp_dy = np.array([0, 0, 0, 1, 1])
print(f'\nReLU\'({x}) = {dy}')
print(f'Expected:      {exp_dy}')
print(f'Derivative: {"PASS" if np.array_equal(dy, exp_dy) else "FAIL"}')

## H2: Sigmoid Numerical Stability

In [ ]:
vals = np.array([-1000, -100, -1, 0, 1, 100, 1000], dtype=float)
results = sigmoid(vals.reshape(1, -1)).flatten()

print('Sigmoid stability:')
for v, r in zip(vals, results):
    ok = 0 <= r <= 1 and not np.isnan(r)
    print(f'  sigmoid({v:>6}) = {r:.10f} | in [0,1]: {0<=r<=1} | {"PASS" if ok else "FAIL"}')

## H3: Softmax Only on Output Layer

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[4, 5, 3, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
x = np.random.randn(3, 4)
output = net.forward(x)

print('Output (softmax):')
for i, row in enumerate(output):
    print(f'  Sample {i}: sum={row.sum():.6f} {"PASS" if abs(row.sum()-1.0)<1e-6 else "FAIL"}')

print('\nHidden activations (should NOT sum to 1):')
for li in range(1, len(net.activations)-1):
    sums = net.activations[li].sum(axis=1)
    is_sm = np.allclose(sums, 1.0)
    print(f'  Layer {li}: sums={sums.round(3)} | {"PASS" if not is_sm else "FAIL (is softmax)"}')

## H4: Dead ReLU Detection

In [ ]:
np.random.seed(42)
config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)

x = np.random.randn(200, 30)
net.forward(x)

print('Dead ReLU analysis (neurons outputting 0 for ALL samples):')
has_problem = False
for i in range(1, len(net.activations)-1):
    act = net.activations[i]
    dead = np.all(act == 0, axis=0).sum()
    total = act.shape[1]
    pct = dead / total * 100
    status = 'WARNING' if pct > 20 else 'OK'
    if pct > 20: has_problem = True
    print(f'  Layer {i}: {dead}/{total} dead ({pct:.1f}%) | {status}')

print(f'\nOverall: {"WARNING" if has_problem else "PASS"}')